In [4]:
# =====================================================================
# STEP 1: INSTALL DEPENDENCIES & PREPARE KAGGLE DATASET
# =====================================================================

!pip install -q transformers evaluate accelerate nltk rouge-score datasets kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
from datasets import Dataset
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM, TrainingArguments, Trainer
import evaluate
import nltk
nltk.download('punkt')

print("Loading IMDB 50K text corpus via KaggleHub...")

# Set the path to the CSV file
file_path = "IMDB Dataset.csv"

# Load the dataset
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews",
  file_path,
)

print("Kaggle Dataset Loaded! First 5 records:")
print(df.head())

# Cleaning and Standardize Columns
df['text'] = df['review']
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})

#Convert Pandas DataFrame to a Hugging Face Dataset object
hf_dataset = Dataset.from_pandas(df[['text', 'label']])

#Create Training and Testing Subsets
#To keep training fast and prevent Colab timeouts, we sample a small subset
hf_dataset = hf_dataset.shuffle(seed=42)
train_subset = hf_dataset.select(range(500))  # 500 rows for training
test_subset = hf_dataset.select(range(500, 600)) # 100 rows for testing

print(f"\nDataset successfully prepared and converted. Train size: {len(train_subset)}, Test size: {len(test_subset)}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/tmp/ipykernel_9665/2001442417.py:25: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Loading IMDB 50K text corpus via KaggleHub...
Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Kaggle Dataset Loaded! First 5 records:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset successfully prepared and converted. Train size: 500, Test size: 100


In [5]:
# =====================================================================
# CELL 2: VARIANT 1 - BERT FINETUNING (FIXED)
# =====================================================================
import time

print("Initializing BERT Sequence Classifier...")

#Load BERT Tokenizer and Model
bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

#Tokenize data with padding and truncation
def tokenize_bert(examples):
    return bert_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train_bert = train_subset.map(tokenize_bert, batched=True)
tokenized_test_bert = test_subset.map(tokenize_bert, batched=True)

#Define the Evaluation Metrics (Precision, Recall, F1)
clf_metrics = evaluate.combine(["precision", "recall", "f1", "accuracy"])
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return clf_metrics.compute(predictions=predictions, references=labels)

#Set up Training Arguments (Updated keyword to 'eval_strategy')
bert_args = TrainingArguments(
    output_dir="./bert_results",
    eval_strategy="epoch",  # <--- FIX APPLIED HERE
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

trainer_bert = Trainer(
    model=bert_model,
    args=bert_args,
    train_dataset=tokenized_train_bert,
    eval_dataset=tokenized_test_bert,
    compute_metrics=compute_metrics,
)

#Train and Evaluate
print("Starting BERT Training...")
start_time = time.time()
trainer_bert.train()
bert_time_per_epoch = time.time() - start_time

bert_eval_results = trainer_bert.evaluate()
print("\n--- BERT Classification Results ---")
print(f"Precision: {bert_eval_results['eval_precision']:.4f}")
print(f"Recall: {bert_eval_results['eval_recall']:.4f}")
print(f"F1-Score: {bert_eval_results['eval_f1']:.4f}")
print(f"Training Time per Epoch: {bert_time_per_epoch:.2f} seconds")

Initializing BERT Sequence Classifier...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.646017,0.593579,0.772727,0.755556,0.764045,0.790000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.646017,0.593579,1,0.772727,0.755556,0.764045,0.790000



--- BERT Classification Results ---
Precision: 0.7727
Recall: 0.7556
F1-Score: 0.7640
Training Time per Epoch: 823.78 seconds


In [7]:
# =====================================================================
# CELL 3: VARIANT 2 - GPT CAUSAL LANGUAGE MODELING
# =====================================================================
print("Initializing GPT Autoregressive Model...")

#Load GPT Tokenizer and Model
gpt_tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
gpt_tokenizer.pad_token = gpt_tokenizer.eos_token
gpt_model = AutoModelForCausalLM.from_pretrained("distilgpt2")

#Tokenize for Causal LM (labels match input_ids)
def tokenize_gpt(examples):
    inputs = gpt_tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    inputs["labels"] = inputs["input_ids"].copy()
    return inputs

tokenized_train_gpt = train_subset.map(tokenize_gpt, batched=True)
tokenized_test_gpt = test_subset.map(tokenize_gpt, batched=True)

#Set up Training Arguments
gpt_args = TrainingArguments(
    output_dir="./gpt_results",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    num_train_epochs=1,
    logging_steps=10,
    report_to="none"
)

trainer_gpt = Trainer(
    model=gpt_model,
    args=gpt_args,
    train_dataset=tokenized_train_gpt,
    eval_dataset=tokenized_test_gpt,
)

#Train and Evaluate (Perplexity)
print("Starting GPT Training...")
start_time = time.time()
trainer_gpt.train()
gpt_time_per_epoch = time.time() - start_time

gpt_eval_results = trainer_gpt.evaluate()
perplexity = np.exp(gpt_eval_results["eval_loss"])

#Generate Sample Text and Calculate BLEU Score
prompt = "This movie was"
input_ids = gpt_tokenizer.encode(prompt, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
gpt_model.to(input_ids.device)
output_ids = gpt_model.generate(input_ids, max_length=30, num_return_sequences=1, pad_token_id=gpt_tokenizer.eos_token_id)
generated_text_gpt = gpt_tokenizer.decode(output_ids[0], skip_special_tokens=True)

bleu_metric = evaluate.load("bleu")

ref_snippet = " ".join(test_subset[0]["text"].split()[:30])
references = [[ref_snippet]]
predictions = [generated_text_gpt]

gpt_bleu = bleu_metric.compute(predictions=predictions, references=references)["bleu"]

print("\n--- GPT Generative Results ---")
print(f"Perplexity: {perplexity:.4f}")
print(f"Generative BLEU Score: {gpt_bleu:.4f}")
print(f"Generated text excerpt: '{generated_text_gpt}'")
print(f"Training Time per Epoch: {gpt_time_per_epoch:.2f} seconds")

Initializing GPT Autoregressive Model...


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Starting GPT Training...


...

Epoch,Training Loss,Validation Loss
1,3.990762,3.802472


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
3.990762,3.802472,1



--- GPT Generative Results ---
Perplexity: 44.8118
Generative BLEU Score: 0.0000
Generated text excerpt: 'This movie was a great movie, and I think it is a great movie. I think it is a great movie. I think it is a great'
Training Time per Epoch: 834.10 seconds


In [8]:
# =====================================================================
# CELL 4: VARIANT 3 - TEXT-GAN FRAMEWORK
# =====================================================================
print("Initializing Adversarial Text-GAN...")

VOCAB_SIZE = 1000
SEQ_LEN = 20
EMBED_DIM = 32

#Define the Generator Network (LSTM)
class TextGenerator(nn.Module):
    def __init__(self):
        super(TextGenerator, self).__init__()
        self.lstm = nn.LSTM(EMBED_DIM, 64, batch_first=True)
        self.fc = nn.Linear(64, VOCAB_SIZE)

    def forward(self, z):
        # Maps latent noise vectors to probability distributions over vocabulary
        out, _ = self.lstm(z)
        logits = self.fc(out)
        return torch.softmax(logits, dim=-1)

#Define the Discriminator Network (1D CNN)
class TextDiscriminator(nn.Module):
    def __init__(self):
        super(TextDiscriminator, self).__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM)
        self.conv = nn.Conv1d(EMBED_DIM, 32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * SEQ_LEN, 1)

    def forward(self, x, is_prob=False):
        if is_prob:
            # Handle soft probabilities generated by the Generator
            embeds = torch.matmul(x, self.embedding.weight)
        else:
            # Handle discrete integer indices from Real Dataset
            embeds = self.embedding(x)

        embeds = embeds.permute(0, 2, 1)
        out = torch.relu(self.conv(embeds))
        out = out.view(out.size(0), -1)
        return torch.sigmoid(self.fc(out))

#Initialize networks, optimizers, and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
netG = TextGenerator().to(device)
netD = TextDiscriminator().to(device)
optD = torch.optim.Adam(netD.parameters(), lr=0.001)
optG = torch.optim.Adam(netG.parameters(), lr=0.001)
criterion = nn.BCELoss()

#Generate Mock Batch (Real Encoded Sentences vs. Latent Noise)
real_data = torch.randint(0, VOCAB_SIZE, (32, SEQ_LEN)).to(device)
noise = torch.randn(32, SEQ_LEN, EMBED_DIM).to(device)

print("Executing Text-GAN training step...")
start_time = time.time()

# --- Train Discriminator ---
optD.zero_grad()
out_real = netD(real_data)
loss_real = criterion(out_real, torch.ones_like(out_real))

fake_probs = netG(noise)
out_fake = netD(fake_probs.detach(), is_prob=True)
loss_fake = criterion(out_fake, torch.zeros_like(out_fake))

loss_D = (loss_real + loss_fake) / 2
loss_D.backward()
optD.step()

# --- Train Generator ---
optG.zero_grad()
out_g = netD(fake_probs, is_prob=True)
loss_G = criterion(out_g, torch.ones_like(out_g)) # Generator wants Discriminator to output 1
loss_G.backward()
optG.step()

gan_time_per_epoch = time.time() - start_time
gan_disc_acc = ((out_real > 0.5).float().mean() + (out_fake < 0.5).float().mean()).item() / 2

#Output Metrics
print("\n--- Text-GAN Adversarial Results ---")
print(f"Discriminator Accuracy: {gan_disc_acc:.4f}")
print(f"Generator Loss: {loss_G.item():.4f}")
print(f"Training Time per Step: {gan_time_per_epoch:.2f} seconds")

Initializing Adversarial Text-GAN...
Executing Text-GAN training step...

--- Text-GAN Adversarial Results ---
Discriminator Accuracy: 0.4688
Generator Loss: 0.6544
Training Time per Step: 0.24 seconds


In [9]:
import json
import pandas as pd
import shutil
from google.colab import files

print("Exporting Project Artifacts...")

# 1. Save Model Pipelines (Weights & Tokenizers)
bert_model.save_pretrained("./models/bert_classifier")
bert_tokenizer.save_pretrained("./models/bert_classifier")

gpt_model.save_pretrained("./models/gpt_generator")
gpt_tokenizer.save_pretrained("./models/gpt_generator")

# 2. Export Training Histories
# Extracting the log history from the Hugging Face Trainers
bert_logs = trainer_bert.state.log_history
gpt_logs = trainer_gpt.state.log_history

# Save metrics to CSV for the repository
metrics_df = pd.DataFrame({
    "Model": ["BERT", "GPT"],
    "Epochs": [1, 1],
    "Eval Loss": [bert_eval_results.get('eval_loss'), gpt_eval_results.get('eval_loss')],
    "F1 Score": [bert_eval_results.get('eval_f1'), None],
    "Perplexity": [None, perplexity],
    "BLEU": [None, gpt_bleu]
})
metrics_df.to_csv("./results/training_metrics.csv", index=False)

# 3. Export Test Inferences
# Save sample generation and classification to JSON
inferences = {
    "bert_sample_classification": {
        "input": test_subset[0]["text"],
        "true_label": test_subset[0]["label"],
        "predicted_label": int(np.argmax(trainer_bert.predict(tokenized_test_bert).predictions[0]))
    },
    "gpt_sample_generation": {
        "prompt": "This movie was",
        "generated_output": generated_text_gpt
    }
}
with open("./results/inference_samples.json", "w") as f:
    json.dump(inferences, f, indent=4)

# 4. Zip and Download the Artifacts
shutil.make_archive("repo_artifacts", 'zip', "./")
files.download("repo_artifacts.zip")
print("Download triggered! Extract this zip into your local repository folder.")

Exporting Project Artifacts...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

OSError: Cannot save file into a non-existent directory: 'results'